# GELAR RASA 2026 ? EDA, preprocessing, CPU modeling, submission

Notebook mandiri: seluruh **gambar kompetisi** dibaca hanya dari dataset GelarRasa yang dipasang. Assignment train dibentuk ulang di sini memakai fitur citra dan sejumlah seed train yang ditinjau tim; tidak membaca NPZ, folder processed, catatan, atau CSV submission lama. Test hanya dipakai saat inferensi. Eksekusi memakai CPU.

In [2]:
from pathlib import Path
import os, re, json, hashlib, time, platform, random, warnings
import numpy as np, pandas as pd
from PIL import Image, ImageOps
import torch, torchvision
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
from sklearn.cluster import MiniBatchKMeans
from scipy.sparse import csr_matrix, diags
from scipy.sparse.csgraph import connected_components

SEED=2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(max(1,min(6,os.cpu_count() or 1)))
DEVICE=torch.device('cpu')
CLASSES=['cardboard','electronic','foodwaste','glass','metal','paper','plastic','textile']
EXTS={'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}
print({'python':platform.python_version(),'torch':torch.__version__,'device':str(DEVICE),'threads':torch.get_num_threads()})

{'python': '3.10.11', 'torch': '2.11.0+cpu', 'device': 'cpu', 'threads': 6}


## 1. Temukan dataset yang dipasang dan periksa strukturnya

Pencarian hanya mencari folder dataset bernama `Dataset DSC GelarRasa 2026` di lokasi umum lokal/Kaggle. Jika tidak ketemu, notebook berhenti dengan daftar path yang terlihat; tidak mengambil data lain.

In [4]:
def find_dataset():
    roots=[Path(r'D:/lomba/NSC UB/GELAR-RASA/data/Dataset DSC GelarRasa 2026'),Path('/kaggle/input'),Path('/content'),Path.cwd()]
    seen=set()
    for root in roots:
        if not root.exists(): continue
        try:
            if root.name=='Dataset DSC GelarRasa 2026': candidates=[root]
            elif str(root).replace('\\','/').endswith('/kaggle/input'):
                candidates=root.rglob('Dataset DSC GelarRasa 2026')
            else: candidates=[]
            for q in candidates:
                q=q.resolve()
                if q in seen: continue
                seen.add(q)
                if (q/'train').is_dir() and (q/'test').is_dir(): return q
        except (PermissionError,OSError): pass
    visible=[]
    for root in roots:
        if root.exists():
            try: visible.append(str(root))
            except Exception: pass
    raise FileNotFoundError('Dataset tidak ditemukan. Path yang dicari: '+json.dumps(visible))
DATA_ROOT=find_dataset(); TRAIN_ROOT=DATA_ROOT/'train'; TEST_ROOT=DATA_ROOT/'test'
# Output bukan input data.
if Path('/kaggle/working').exists(): OUTPUT_ROOT=Path('/kaggle/working/gelarrasa_cpu_run')
else: OUTPUT_ROOT=DATA_ROOT.parents[1]/'final-submit'/'output'/'cpu_raw_dataset_run'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
def image_files(folder): return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXTS],key=lambda p:int(re.search(r'\d+',p.stem).group()) if re.search(r'\d+',p.stem) else p.stem)
train_paths=image_files(TRAIN_ROOT); test_paths=image_files(TEST_ROOT)
assert len(train_paths)==1920, f'Diharapkan 1920 train, ditemukan {len(train_paths)} di {TRAIN_ROOT}'
assert len(test_paths)==480, f'Diharapkan 480 test, ditemukan {len(test_paths)} di {TEST_ROOT}'
def sid(p):
    m=re.search(r'\d+',p.stem); assert m, f'ID tidak ditemukan: {p.name}'; return m.group()
train_ids=[sid(p) for p in train_paths]; test_ids=[sid(p) for p in test_paths]
assert len(set(train_ids))==len(train_ids) and len(set(test_ids))==len(test_ids)
# Numeric IDs are local to each split and may overlap across train/test.
print('DATA_ROOT:',DATA_ROOT,'| train:',len(train_paths),'| test:',len(test_paths),'| output:',OUTPUT_ROOT)

DATA_ROOT: D:\lomba\NSC UB\GELAR-RASA\data\Dataset DSC GelarRasa 2026 | train: 1920 | test: 480 | output: D:\lomba\NSC UB\GELAR-RASA\final-submit\output\cpu_raw_dataset_run


## 2. EDA dan pemeriksaan file

Catat dimensi, mode, format, ukuran berkas dan hash setiap foto. Hash hanya menjadi identitas sumber; tidak mengakses label eksternal.

In [6]:
def inspect(paths,split):
    rows=[]
    for p in paths:
        try:
            with Image.open(p) as im:
                im=ImageOps.exif_transpose(im); im.load()
                rows.append({'split':split,'image_id':sid(p),'path':str(p),'width':im.width,'height':im.height,'mode':im.mode,'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'error':None})
        except Exception as e: rows.append({'split':split,'image_id':sid(p),'path':str(p),'error':repr(e)})
    return rows
inventory=pd.DataFrame(inspect(train_paths,'train')+inspect(test_paths,'test'))
assert inventory.error.isna().all(), inventory[inventory.error.notna()].head()
print(inventory.groupby('split')[['width','height','bytes']].agg(['min','median','max']).round(1))
print('duplicate pixel hashes:',int(inventory.duplicated('sha256').sum()))
inventory.to_csv(OUTPUT_ROOT/'image_inventory.csv',index=False)

      width              height              bytes                  
        min median   max    min median   max   min   median      max
split                                                               
test    129  416.0  1600    168  416.0  1167  3114  22512.5   885324
train   100  509.0  2605    100  485.5  4032  2132  24924.5  2689277
duplicate pixel hashes: 4


## 3. Seed tim dan penetapan assignment train-only

Seed yang ditinjau dipakai sebagai titik awal graf kemiripan **train saja**. Semua gambar train lainnya memperoleh label melalui propagasi label pada tetangga train. Jumlah seed per kelas dibatasi maksimum lima. Test tidak masuk pembentukan graf, pemilihan parameter, atau pelatihan.

In [8]:
SEED_LABELS={
 '0343':'paper','1817':'paper','0352':'paper','0396':'paper','0429':'paper',
 '1115':'plastic','0200':'foodwaste','0265':'foodwaste','1039':'foodwaste',
 '1215':'cardboard','0536':'cardboard','0086':'electronic','0512':'electronic',
 '0637':'metal','0714':'metal','0267':'glass','0251':'glass','0615':'textile'
}
assert set(SEED_LABELS)<=set(train_ids)
seed_counts=pd.Series(SEED_LABELS).value_counts()
assert set(SEED_LABELS.values())<=set(CLASSES) and (seed_counts<=5).all()
print('seed total:',len(SEED_LABELS),'| per class:',seed_counts.reindex(CLASSES,fill_value=0).to_dict())

seed total: 18 | per class: {'cardboard': 2, 'electronic': 2, 'foodwaste': 3, 'glass': 2, 'metal': 2, 'paper': 5, 'plastic': 1, 'textile': 1}


## 4. Preprocessing dan ekstraksi fitur CPU

Foto diorientasikan menurut EXIF, diubah ke RGB, letterbox ke 224?224. Fitur gabungan memakai MobileNetV3-Large pretrained beku ditambah histogram warna HSV dan HOG. Bobot ImageNet berasal dari torchvision; tidak ada model vision-language.

In [10]:
from skimage.feature import hog
from skimage.color import rgb2hsv
from torchvision.transforms import functional as TF
weights=MobileNet_V3_Large_Weights.DEFAULT
backbone=mobilenet_v3_large(weights=weights).features.eval().to(DEVICE)
preprocess=weights.transforms()
@torch.inference_mode()
def cnn_one(path):
    with Image.open(path) as im: im=ImageOps.exif_transpose(im).convert('RGB')
    im.thumbnail((224,224),Image.Resampling.LANCZOS)
    canvas=Image.new('RGB',(224,224),(127,127,127)); canvas.paste(im,((224-im.width)//2,(224-im.height)//2))
    x=preprocess(canvas).unsqueeze(0).to(DEVICE)
    f=backbone(x).mean(dim=(2,3)).flatten().cpu().numpy().astype('float32')
    return f

def hand_one(path):
    with Image.open(path) as im: im=ImageOps.exif_transpose(im).convert('RGB').resize((96,96),Image.Resampling.LANCZOS)
    a=np.asarray(im)
    hsv=rgb2hsv(a)
    color=np.concatenate([np.histogram(hsv[:,:,i],bins=16,range=(0,1),density=False)[0] for i in range(3)]).astype('float32')
    color/=max(1,color.sum())
    gray=np.asarray(im.convert('L'),dtype='float32')/255
    texture=hog(gray,orientations=9,pixels_per_cell=(8,8),cells_per_block=(2,2),feature_vector=True).astype('float32')
    return np.concatenate([normalize(color.reshape(1,-1))[0],normalize(texture.reshape(1,-1))[0]])
all_paths=train_paths+test_paths; all_ids=train_ids+test_ids
started=time.time(); cnn=[]; hand=[]
for i,p in enumerate(all_paths):
    cnn.append(cnn_one(p)); hand.append(hand_one(p))
    if (i+1)%200==0: print(f'{i+1}/{len(all_paths)} fitur selesai; {time.time()-started:.1f}s')
Xcnn=normalize(np.stack(cnn)); Xhand=normalize(np.stack(hand))
X=normalize(np.concatenate([Xcnn*.8,Xhand*.2],axis=1))
X_train=X[:len(train_paths)]; X_test=X[len(train_paths):]
np.savez_compressed(OUTPUT_ROOT/'features_cpu.npz',train_ids=np.asarray(train_ids),test_ids=np.asarray(test_ids),X_train=X_train,X_test=X_test)
print('feature matrix:',X_train.shape,X_test.shape,'elapsed_s:',round(time.time()-started,1))

200/2400 fitur selesai; 11.2s
400/2400 fitur selesai; 22.5s
600/2400 fitur selesai; 33.8s
800/2400 fitur selesai; 44.8s
1000/2400 fitur selesai; 56.3s
1200/2400 fitur selesai; 67.8s
1400/2400 fitur selesai; 78.9s
1600/2400 fitur selesai; 90.5s
1800/2400 fitur selesai; 101.5s
2000/2400 fitur selesai; 112.3s
2200/2400 fitur selesai; 123.5s
2400/2400 fitur selesai; 133.4s
feature matrix: (1920, 5364) (480, 5364) elapsed_s: 137.3


## 5. Assignment train dengan graf kNN

Graf cosine kNN dibuat hanya pada fitur train. Seed dikunci pada label yang ditinjau; label sisanya diperoleh dari difusi label berulang. Ini pseudo-label algoritmis, bukan label resmi independen. Simpan distribusi dan confidence agar mudah diaudit.

In [12]:
N_NEIGHBORS=15
nn=NearestNeighbors(n_neighbors=N_NEIGHBORS+1,metric='cosine',n_jobs=-1).fit(X_train)
dist,idx=nn.kneighbors(X_train)
rows=np.repeat(np.arange(len(train_ids)),N_NEIGHBORS); cols=idx[:,1:].reshape(-1); ds=dist[:,1:].reshape(-1)
weights_graph=np.exp(-np.maximum(ds,0)/.12)
W=csr_matrix((weights_graph,(rows,cols)),shape=(len(train_ids),len(train_ids)))
W=W.maximum(W.T); W.setdiag(0); W.eliminate_zeros()
# Symmetric normalized propagation, with seed clamping.
labels_to_i={c:i for i,c in enumerate(CLASSES)}
Y=np.zeros((len(train_ids),len(CLASSES)),dtype='float64')
seed_pos={train_ids.index(k):labels_to_i[v] for k,v in SEED_LABELS.items()}
for j,c in seed_pos.items(): Y[j,c]=1
D=np.asarray(W.sum(axis=1)).ravel(); inv=np.where(D>0,1/np.sqrt(np.maximum(D,1e-12)),0)
S=diags(inv)@W@diags(inv)
F=Y.copy(); alpha=.92
for _ in range(80):
    F=alpha*(S@F)+(1-alpha)*Y
    for j,c in seed_pos.items(): F[j,:]=0; F[j,c]=1
prob=F/np.maximum(F.sum(axis=1,keepdims=True),1e-12)
assigned=prob.argmax(1); confidence=prob.max(1)
for j,c in seed_pos.items(): assigned[j]=c; confidence[j]=1.
train_labels=np.asarray(CLASSES)[assigned]
assignment=pd.DataFrame({'image_id':train_ids,'assigned_class':train_labels,'confidence':confidence,'human_seed':[i in seed_pos for i in range(len(train_ids))],'source_sha256':inventory.query("split=='train'").sha256.to_numpy()})
print(assignment.assigned_class.value_counts().reindex(CLASSES,fill_value=0))
print('confidence summary:',assignment.confidence.describe().round(3).to_dict())
assignment.to_csv(OUTPUT_ROOT/'train_assignments.csv',index=False)
assert assignment.assigned_class.isin(CLASSES).all()

assigned_class
cardboard     283
electronic    151
foodwaste     334
glass         194
metal         165
paper         513
plastic        88
textile       192
Name: count, dtype: int64
confidence summary: {'count': 1920.0, 'mean': 0.474, 'std': 0.213, 'min': 0.165, '25%': 0.306, '50%': 0.423, '75%': 0.588, 'max': 1.0}


## 6. Grouped proxy validation dan model CPU

Source group dibuat dari perceptual hash sederhana. Model dibandingkan memakai grouped folds; metrik mengukur agreement dengan assignment algoritmis, bukan akurasi ground truth. Model final dilatih pada seluruh assignment train.

In [14]:
from scipy.fft import dctn

def phash(path):
    with Image.open(path) as im: im=ImageOps.exif_transpose(im).convert('L').resize((32,32),Image.Resampling.LANCZOS)
    z=dctn(np.asarray(im,dtype='float32'),norm='ortho')[:8,:8].ravel()[1:]
    return z>np.median(z)
ph=np.stack([phash(p) for p in train_paths])
# Deterministic connected components for near-duplicate sources (Hamming <= 3).
parent=np.arange(len(train_ids))
def find(a):
    while parent[a]!=a: parent[a]=parent[parent[a]]; a=parent[a]
    return a
def union(a,b):
    a,b=find(a),find(b)
    if a!=b: parent[max(a,b)]=min(a,b)
for i in range(len(ph)):
    if i+1<len(ph):
        d=np.count_nonzero(ph[i+1:]!=ph[i],axis=1)
        for j in np.flatnonzero(d<=3)+i+1: union(i,int(j))
groups=np.asarray([find(i) for i in range(len(ph))]).astype(str)
model=SVC(C=4,kernel='rbf',gamma='scale',decision_function_shape='ovr',probability=True,random_state=SEED)
# grouped proxy validation, with adaptive fold count for available groups
n_groups=len(np.unique(groups)); n_splits=min(5,n_groups)
cv=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=SEED)
oof=np.empty(len(train_ids),dtype=object); fold_rows=[]
for fold,(a,b) in enumerate(cv.split(X_train,train_labels,groups)):
    model_fold=SVC(C=4,kernel='rbf',gamma='scale',decision_function_shape='ovr')
    model_fold.fit(X_train[a],train_labels[a],sample_weight=np.maximum(.1,confidence[a]))
    pred=model_fold.predict(X_train[b]); oof[b]=pred
    fold_rows.append({'fold':fold,'n_train':len(a),'n_valid':len(b),'agreement_to_algorithmic_assignment':accuracy_score(train_labels[b],pred),'macro_f1':f1_score(train_labels[b],pred,labels=CLASSES,average='macro',zero_division=0),'balanced_accuracy':balanced_accuracy_score(train_labels[b],pred)})
fold_table=pd.DataFrame(fold_rows); print(fold_table); print('mean proxy agreement:',round(fold_table.agreement_to_algorithmic_assignment.mean(),4))
model.fit(X_train,train_labels,sample_weight=np.maximum(.1,confidence))

   fold  n_train  ...  macro_f1  balanced_accuracy
0     0     1532  ...  0.853783           0.848375
1     1     1538  ...  0.878234           0.876750
2     2     1537  ...  0.810699           0.791507
3     3     1538  ...  0.873708           0.859446
4     4     1535  ...  0.863063           0.852605

[5 rows x 6 columns]
mean proxy agreement: 0.874


## 7. Inferensi test dan file submission

ID test mengikuti urutan numerik file di folder test. Test tidak ikut fitting, graph propagation, atau pemilihan parameter.

In [16]:
pred=model.predict(X_test)
submission=pd.DataFrame({'image_id':test_ids,'waste_category':pred})
assert len(submission)==480 and submission.image_id.is_unique and submission.waste_category.isin(CLASSES).all()
submission_path=OUTPUT_ROOT/'DSC2026_Submission_CPU.csv'
submission.to_csv(submission_path,index=False)
check=pd.read_csv(submission_path,dtype={'image_id':str})
assert check.equals(submission)
print('submission:',submission_path,'| rows:',len(submission),'| counts:',submission.waste_category.value_counts().to_dict())

submission: D:\lomba\NSC UB\GELAR-RASA\final-submit\output\cpu_raw_dataset_run\DSC2026_Submission_CPU.csv | rows: 480 | counts: {'paper': 169, 'glass': 66, 'cardboard': 64, 'foodwaste': 59, 'electronic': 35, 'metal': 34, 'textile': 33, 'plastic': 20}


## 8. Paket keluaran dan batas interpretasi

Run menghasilkan inventory citra, fitur train/test, assignment algoritmis train, proxy validation, dan CSV submission. Label train bukan ground truth resmi; hasil tidak dijamin sama dengan submission lama atau mendekati selisih tertentu tanpa memasukkan target lama sebagai sumber.

In [18]:
run_manifest={
 'seed':SEED,'device':'cpu','dataset_root':str(DATA_ROOT),'train_images':len(train_ids),'test_images':len(test_ids),
 'seed_count':len(SEED_LABELS),'seed_class_counts':seed_counts.reindex(CLASSES,fill_value=0).to_dict(),
 'train_assignment_counts':assignment.assigned_class.value_counts().reindex(CLASSES,fill_value=0).to_dict(),
 'proxy_validation_mean_agreement':float(fold_table.agreement_to_algorithmic_assignment.mean()),
 'proxy_validation_scope':'agreement with train-only propagated assignments; not official accuracy',
 'submission_path':str(submission_path),'submission_sha256':hashlib.sha256(submission_path.read_bytes()).hexdigest(),
 'input_sources':'competition dataset images only; no local cache, processed files, historical labels, or old submissions',
 'pretrained_backbone':'torchvision MobileNetV3-Large ImageNet weights; frozen feature extractor'
}
(OUTPUT_ROOT/'run_manifest.json').write_text(json.dumps(run_manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(run_manifest,indent=2,ensure_ascii=False))

{
  "seed": 2026,
  "device": "cpu",
  "dataset_root": "D:\\lomba\\NSC UB\\GELAR-RASA\\data\\Dataset DSC GelarRasa 2026",
  "train_images": 1920,
  "test_images": 480,
  "seed_count": 18,
  "seed_class_counts": {
    "cardboard": 2,
    "electronic": 2,
    "foodwaste": 3,
    "glass": 2,
    "metal": 2,
    "paper": 5,
    "plastic": 1,
    "textile": 1
  },
  "train_assignment_counts": {
    "cardboard": 283,
    "electronic": 151,
    "foodwaste": 334,
    "glass": 194,
    "metal": 165,
    "paper": 513,
    "plastic": 88,
    "textile": 192
  },
  "proxy_validation_mean_agreement": 0.873950028953946,
  "proxy_validation_scope": "agreement with train-only propagated assignments; not official accuracy",
  "submission_path": "D:\\lomba\\NSC UB\\GELAR-RASA\\final-submit\\output\\cpu_raw_dataset_run\\DSC2026_Submission_CPU.csv",
  "submission_sha256": "2052db4f010b9619064f72c6e31a6fb35b53a8fa21508b4c91b54ab45c035d8f",
  "input_sources": "competition dataset images only; no local cache,